# SupportOps AI - Classical NLP Model Tuning

## Objective

Improve the BANKING77 TF-IDF + Logistic Regression baseline using a
validation-based hyperparameter tuning strategy.

The official BANKING77 test set is kept untouched during model selection.

### Workflow

Official Training Data
→ Train / Validation Split
→ Hyperparameter Search
→ Best Model Selection
→ Retrain on Full Training Data
→ Official Test Evaluation
→ Strict Zero-Overlap Evaluation

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold
)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
TRAIN_PATH = "../data/raw/banking77_train.csv"
TEST_PATH = "../data/raw/banking77_test.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Official training shape:", train_df.shape)
print("Official testing shape:", test_df.shape)

train_df.head()

Official training shape: (10003, 2)
Official testing shape: (3080, 2)


,text,intent
0,I am still waiting on my card?,card_arrival
1,What can I do if my card still hasn't arrived ...,card_arrival
2,I have been waiting over a week. Is the card s...,card_arrival
3,Can I track my card while it is in the process...,card_arrival
4,"How do I know if I will get my card, or if it ...",card_arrival


Recreate the strict test set

In [3]:
def normalize_text(text):
    return " ".join(
        str(text)
        .lower()
        .strip()
        .split()
    )

In [4]:
train_df["normalized_text"] = (
    train_df["text"].apply(normalize_text)
)

test_df["normalized_text"] = (
    test_df["text"].apply(normalize_text)
)

train_normalized_texts = set(
    train_df["normalized_text"]
)

strict_test_df = test_df[
    ~test_df["normalized_text"].isin(
        train_normalized_texts
    )
].copy()

print("Official test:", len(test_df))
print("Strict test:", len(strict_test_df))

Official test: 3080
Strict test: 3073


In [5]:
strict_overlap = set(
    train_df["normalized_text"]
).intersection(
    set(strict_test_df["normalized_text"])
)

print("Strict overlap:", len(strict_overlap))

Strict overlap: 0


Create a train/validation split

In [6]:
X = train_df["text"]
y = train_df["intent"]

In [7]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

In [8]:
print("Training subset:", len(X_train))
print("Validation subset:", len(X_val))

print(
    "Number of training intents:",
    y_train.nunique()
)

print(
    "Number of validation intents:",
    y_val.nunique()
)

Training subset: 8502
Validation subset: 1501
Number of training intents: 77
Number of validation intents: 77


Create the tuning pipeline

In [9]:
pipeline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            random_state=42
        )
    )
])

Define the hyperparameter search space

In [10]:
param_grid = {
    "tfidf__ngram_range": [
        (1, 1),
        (1, 2)
    ],
    
    "tfidf__min_df": [
        1,
        2,
        3
    ],
    
    "tfidf__sublinear_tf": [
        True,
        False
    ],
    
    "classifier__C": [
        0.5,
        1.0,
        2.0,
        4.0
    ]
}

Create stratified cross-validation

In [11]:
cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

Configure GridSearchCV

In [12]:
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

Run the hyperparameter search

In [13]:
grid_search.fit(
    X_train,
    y_train
)

Fitting 3 folds for each of 48 candidates, totalling 144 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'classifier__C': [0.5, 1.0, ...], 'tfidf__min_df': [1, 2, ...], 'tfidf__ngram_range': [(1, ...), (1, ...)], 'tfidf__sublinear_tf': [True, False]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1_macro'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, t

Find the best configuration

In [14]:
print(
    "Best CV Macro F1:",
    round(
        grid_search.best_score_,
        4
    )
)

print("\nBest parameters:")

for parameter, value in (
    grid_search.best_params_.items()
):
    print(
        f"{parameter}: {value}"
    )

Best CV Macro F1: 0.8667

Best parameters:
classifier__C: 4.0
tfidf__min_df: 1
tfidf__ngram_range: (1, 1)
tfidf__sublinear_tf: True


Inspect the tuning results

In [15]:
cv_results = pd.DataFrame(
    grid_search.cv_results_
)

In [16]:
tuning_results = cv_results[
    [
        "param_tfidf__ngram_range",
        "param_tfidf__min_df",
        "param_tfidf__sublinear_tf",
        "param_classifier__C",
        "mean_train_score",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].sort_values(
    "rank_test_score"
)

tuning_results.head(10)

,param_tfidf__ngram_range,param_tfidf__min_df,param_tfidf__sublinear_tf,param_classifier__C,mean_train_score,mean_test_score,std_test_score,rank_test_score
36,"(1, 1)",1,True,4.0,0.981481,0.866661,0.003126,1
37,"(1, 1)",1,False,4.0,0.981186,0.865987,0.004405,2
40,"(1, 1)",2,True,4.0,0.977046,0.864088,0.001331,3
41,"(1, 1)",2,False,4.0,0.976862,0.861957,0.004020,4
44,"(1, 1)",3,True,4.0,0.974893,0.861747,0.003076,5
45,"(1, 1)",3,False,4.0,0.974459,0.859636,0.004145,6
24,"(1, 1)",1,True,2.0,0.959942,0.858919,0.006543,7
25,"(1, 1)",1,False,2.0,0.959446,0.858139,0.005848,8
38,"(1, 2)",1,True,4.0,0.995577,0.857359,0.010323,9
28,"(1, 1)",2,True,2.0,0.955237,0.857354,0.006248,10


Evaluate best model on our held-out validation set

In [17]:
best_cv_model = (
    grid_search.best_estimator_
)

In [18]:
val_pred = best_cv_model.predict(
    X_val
)

In [19]:
val_accuracy = accuracy_score(
    y_val,
    val_pred
)

val_f1 = f1_score(
    y_val,
    val_pred,
    average="macro",
    zero_division=0
)

print(
    f"Validation Accuracy: {val_accuracy:.4f}"
)

print(
    f"Validation Macro F1: {val_f1:.4f}"
)

Validation Accuracy: 0.8867
Validation Macro F1: 0.8853


Compare baseline vs tuned model

In [21]:
original_baseline = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            random_state=42
        )
    )
])

In [22]:
original_baseline.fit(
    X_train,
    y_train
)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidf', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](77,)","['Refund_not_showing_up','activate_my_card','age_limit',..., 'why_verify_identity','wrong_amount_of_cash_received', 'wrong_exchange_rate_for_cash_withdrawal']"
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"max_df max_df: float or int, default=1.0When building the vocabulary ignore terms that have a documentfrequency strictly higher than the given threshold (corpus-specificstop words).If float in range [0.0, 1.0], the parameter represents a proportion ofdocuments, integer absolute counts.This parameter is ignored if vocabulary is not None.",0.95
,"min_df min_df: float or int, default=1When building the vocabulary ignore terms that have a documentfrequency strictly lower than the given threshold. This value is alsocalled cut-off in the literature.If float in range of [0.0, 1.0], the parameter represents a proportionof documents, integer absolute counts.This parameter is ignored if vocabulary is not None.",2
,"sublinear_tf sublinear_tf: bool, default=FalseApply sublinear tf scaling, i.e. replace tf with 1 + log(tf).",True
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'


In [23]:
original_val_pred = original_baseline.predict(
    X_val
)

In [24]:
original_val_f1 = f1_score(
    y_val,
    original_val_pred,
    average="macro",
    zero_division=0
)

print(
    f"Original Baseline Validation Macro F1: "
    f"{original_val_f1:.4f}"
)

print(
    f"Tuned Model Validation Macro F1: "
    f"{val_f1:.4f}"
)

Original Baseline Validation Macro F1: 0.8561
Tuned Model Validation Macro F1: 0.8853


In [25]:
comparison = pd.DataFrame({
    "Model": [
        "Original TF-IDF + Logistic Regression",
        "Tuned TF-IDF + Logistic Regression"
    ],
    "Macro F1": [
        original_val_f1,
        val_f1
    ]
})

comparison

,Model,Macro F1
0,Original TF-IDF + Logistic Regression,0.856087
1,Tuned TF-IDF + Logistic Regression,0.885270


Now retrain the best configuration on ALL official training data

In [26]:
final_baseline_model = (
    grid_search.best_estimator_
)

final_baseline_model.fit(
    train_df["text"],
    train_df["intent"]
)

print(
    "Final tuned baseline trained on "
    "all official training data!"
)

Final tuned baseline trained on all official training data!


Final official test evaluation

In [27]:
official_pred = (
    final_baseline_model.predict(
        test_df["text"]
    )
)

In [28]:
official_accuracy = accuracy_score(
    test_df["intent"],
    official_pred
)

official_precision = precision_score(
    test_df["intent"],
    official_pred,
    average="macro",
    zero_division=0
)

official_recall = recall_score(
    test_df["intent"],
    official_pred,
    average="macro",
    zero_division=0
)

official_f1 = f1_score(
    test_df["intent"],
    official_pred,
    average="macro",
    zero_division=0
)

In [29]:
print("FINAL TUNED BASELINE — OFFICIAL TEST")
print("=" * 50)

print(
    f"Accuracy:        {official_accuracy:.4f}"
)

print(
    f"Macro Precision: {official_precision:.4f}"
)

print(
    f"Macro Recall:    {official_recall:.4f}"
)

print(
    f"Macro F1:        {official_f1:.4f}"
)

FINAL TUNED BASELINE — OFFICIAL TEST
Accuracy:        0.9019
Macro Precision: 0.9068
Macro Recall:    0.9019
Macro F1:        0.9022


Evaluate strict test set

In [30]:
strict_pred = (
    final_baseline_model.predict(
        strict_test_df["text"]
    )
)

In [31]:
strict_accuracy = accuracy_score(
    strict_test_df["intent"],
    strict_pred
)

strict_f1 = f1_score(
    strict_test_df["intent"],
    strict_pred,
    average="macro",
    zero_division=0
)

print("STRICT ZERO-OVERLAP TEST")
print("=" * 50)

print(
    f"Accuracy: {strict_accuracy:.4f}"
)

print(
    f"Macro F1: {strict_f1:.4f}"
)

STRICT ZERO-OVERLAP TEST
Accuracy: 0.9017
Macro F1: 0.9020


Build the model comparison table

In [32]:
final_results = pd.DataFrame({
    "Evaluation": [
        "Official BANKING77",
        "Strict Zero-Overlap"
    ],
    "Accuracy": [
        official_accuracy,
        strict_accuracy
    ],
    "Macro F1": [
        official_f1,
        strict_f1
    ]
})

final_results

,Evaluation,Accuracy,Macro F1
0,Official BANKING77,0.901948,0.902169
1,Strict Zero-Overlap,0.901725,0.901983


Save the tuned classical model

In [33]:
import joblib

In [34]:
from pathlib import Path

MODEL_DIR = Path(
    "../ml/artifacts"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [35]:
MODEL_PATH = (
    MODEL_DIR
    / "banking77_tfidf_logreg.joblib"
)

joblib.dump(
    final_baseline_model,
    MODEL_PATH
)

print(
    "Model saved to:",
    MODEL_PATH
)

Model saved to: ../ml/artifacts/banking77_tfidf_logreg.joblib


Test the saved model

In [36]:
loaded_model = joblib.load(
    MODEL_PATH
)

In [37]:
test_message = (
    "I was charged twice for the same card payment"
)

prediction = loaded_model.predict(
    [test_message]
)[0]

print(
    "Prediction:",
    prediction
)

Prediction: transaction_charged_twice


Save tuning results

In [38]:
RESULTS_DIR = Path(
    "../ml/evaluation"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [39]:
tuning_results.to_csv(
    RESULTS_DIR
    / "banking77_baseline_tuning_results.csv",
    index=False
)

final_results.to_csv(
    RESULTS_DIR
    / "banking77_baseline_final_results.csv",
    index=False
)

print(
    "Experiment results saved."
)

Experiment results saved.


Final summary cell

In [40]:
print("=" * 65)
print("SUPPORTOPS AI - TUNED CLASSICAL BASELINE")
print("=" * 65)

print("\nBest Hyperparameters:")

for parameter, value in (
    grid_search.best_params_.items()
):
    print(
        f"{parameter}: {value}"
    )

print(
    f"\nBest CV Macro F1: "
    f"{grid_search.best_score_:.4f}"
)

print(
    f"Validation Macro F1: "
    f"{val_f1:.4f}"
)

print("\nFINAL OFFICIAL TEST")
print(
    f"Accuracy: {official_accuracy:.4f}"
)
print(
    f"Macro F1: {official_f1:.4f}"
)

print("\nSTRICT TEST")
print(
    f"Accuracy: {strict_accuracy:.4f}"
)
print(
    f"Macro F1: {strict_f1:.4f}"
)

print(
    f"\nStrict overlap: {len(strict_overlap)}"
)

print(
    "\nClassical baseline tuning complete!"
)

SUPPORTOPS AI - TUNED CLASSICAL BASELINE

Best Hyperparameters:
classifier__C: 4.0
tfidf__min_df: 1
tfidf__ngram_range: (1, 1)
tfidf__sublinear_tf: True

Best CV Macro F1: 0.8667
Validation Macro F1: 0.8853

FINAL OFFICIAL TEST
Accuracy: 0.9019
Macro F1: 0.9022

STRICT TEST
Accuracy: 0.9017
Macro F1: 0.9020

Strict overlap: 0

Classical baseline tuning complete!
